# 文献检索与管理：找文献、用 Zotero、引用格式与快速读论文

本笔记本由文章自动生成，代码和[原文](https://wxrkz123.github.io/blog/Literature-Management/)完全一致。文中的解释、公式、图和每段代码的输出，请对照原文阅读。

## 五、检查 `.bib` 文件

投稿之前，可以用一小段代码检查 `.bib` 文件：有没有重复的条目，期刊论文有没有缺少卷、期、页码、DOI。下面用一个故意包含几处问题的示例文件演示（用到[正则表达式](https://wxrkz123.github.io/blog/REGEX-01-Basics/)）：

In [ ]:
import re
from collections import Counter

bib = r"""
@article{card1994,
  author  = {Card, David and Krueger, Alan B.},
  title   = {Minimum Wages and Employment: A Case Study of the Fast-Food Industry in {New Jersey} and {Pennsylvania}},
  journal = {American Economic Review},
  year    = {1994}, volume = {84}, number = {4}, pages = {772--793},
}
@article{bertrand2004,
  author  = {Bertrand, Marianne and Duflo, Esther and Mullainathan, Sendhil},
  title   = {How Much Should We Trust Differences-in-Differences Estimates?},
  journal = {Quarterly Journal of Economics},
  year    = {2004}, volume = {119}, number = {1}, pages = {249--275},
  doi     = {10.1162/003355304772839588},
}
@article{bertrand2004a,
  author  = {Bertrand, M. and Duflo, E. and Mullainathan, S.},
  title   = {How much should we trust differences-in-differences estimates},
  journal = {The Quarterly Journal of Economics},
  year    = {2004},
}
@book{angrist2009,
  author    = {Angrist, Joshua D. and Pischke, J{\"o}rn-Steffen},
  title     = {Mostly Harmless Econometrics: An Empiricist's Companion},
  publisher = {Princeton University Press},
  year      = {2009},
}
"""

entries = []
for m in re.finditer(r"@(\w+)\{([^,]+),(.*?)\n\}", bib, re.S):
    kind, key, body = m.group(1).lower(), m.group(2).strip(), m.group(3)
    fields = {k.lower(): v.strip() for k, v in re.findall(r"(\w+)\s*=\s*\{((?:[^{}]|\{[^{}]*\})*)\}", body)}
    entries.append((kind, key, fields))
print(f"共 {len(entries)} 条文献")

required = {"article": ["author", "title", "journal", "year", "volume", "pages", "doi"],
            "book": ["author", "title", "publisher", "year"]}
for kind, key, fields in entries:
    missing = [f for f in required.get(kind, []) if f not in fields]
    if missing:
        print(f"  {key}（{kind}）缺少：{', '.join(missing)}")

def normalize(title):
    """去掉大小写、标点和花括号，用于发现重复的条目"""
    return re.sub(r"[^a-z0-9]", "", title.lower())

dup = Counter(normalize(f["title"]) for _, _, f in entries)
for kind, key, fields in entries:
    if dup[normalize(fields["title"])] > 1:
        print(f"  可能重复：{key} —— {fields['title'][:50]}")